# Neighbourhood Characteristics and House Prices

Our goal is to explore which neighbourhood characteristics are most strongly associated with house prices in London. By analyzing data at the Lower Layer Super Output Area (LSOA) level, we have roughly 4,800 neighbourhoods to compare, allowing for robust statistical testing and multiple linear regression.

## Task 1: Identify a real-world problem
**Problem Statement:** Which neighbourhood characteristics (crime, public transport accessibility, deprivation, and school quality) are associated with house prices in London? Can we identify which areas offer the best value for money based on these metrics?

**Stakeholders:** Homebuyers, real estate developers, urban planners, and local authorities.

## Task 2: Data Selection
**Datasets Chosen & Sources:**
1. **Median House Prices (ONS):** Contains median house prices paid at the LSOA level. We chose this over borough-level averages because it prevents extreme luxury properties (outliers) from skewing the data. Timeframe: 1995 to 2023. `HPSSA Dataset 46 - Median price paid for residential properties by LSOA.xls` [https://www.ons.gov.uk/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46](https://www.ons.gov.uk/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46)
2. **MPS LSOA Level Crime (London Datastore):** Granular crime counts by category at the LSOA level. Allows us to test if specific crime types (e.g., burglary vs. anti-social behaviour) correlate differently with property values. Timeframe: July 2020 to June 2024. `MPS LSOA Level Crime (Historical).csv` [https://data.london.gov.uk/dataset/recorded_crime_summary](https://data.london.gov.uk/download/exy3m/vm7/MPS%20LSOA%20Level%20Crime%20(Historical).csv)
3. **Index of Multiple Deprivation (IMD 2019):** An official government measure of relative deprivation. Captures crucial socio-economic health variables (income, employment, health, education) in a single unified index. Timeframe: 2019. `File7:all ranks, deciles and scores for the indices of deprication, and population denominators` [https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019](https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv)
4. **Public Transport Accessibility Levels (PTAL):** Measures public transport connectivity. Crucial for London, where transport links heavily dictate property desirability. Timeframe: 2015. `2015 PTAL LSOA 2011` [https://data.london.gov.uk/dataset/public-transport-accessibility-levels](https://data.london.gov.uk/download/24rz6/77d9b319-931e-4090-bf8e-f578938bd352/LSOA2011%20AvPTAI2015.csv)

**Justification for Approach:** 
By shifting to the Lower Layer Super Output Area (LSOA) level, we transform a basic 33-point borough analysis into a rich, complex 4,800-point dataset. This granularity allows for rigorous multiple linear regression, letting us isolate the specific impact of transport, crime, and deprivation on house prices.

**Navigating Real-World Data Limitations (Potential Issues):** 
While this dataset is incredibly rich, it comes with several real-world complexities that we expect to handle in our cleaning phase:
*   **Temporal Misalignment:** IMD is from 2019, PTAL is from 2015, and our Crime/Price data will be focused on 2023. We must assume that relative deprivation and transport infrastructure change slowly over time.
*   **Geographic Mismatches:** We will need to perform complex dataframe merges on `LSOA_Code`. It is highly likely that some datasets include areas outside of London or have missing codes that will result in missing data (`NaN`) after merging.
*   **Aggregating Time-Series to Cross-Sectional:** The crime data is monthly; we will need to aggregate this into annual totals to match our target year.

## Task 3: Data Loading

In [1]:
from pathlib import Path
import json
import csv
import urllib.request
import urllib.parse
from collections import Counter, defaultdict
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from zipfile import ZipFile
import urllib
import json
from collections import Counter, defaultdict

In [2]:
# Set file paths

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def download_if_needed(url, path):
    if not path.exists():
        print(f"Downloading {path.name}...")
        r = requests.get(url)
        r.raise_for_status()
        path.write_bytes(r.content)
    else:
        print(f"{path.name} already exists.")

def extract_if_needed(zip_path, extract_folder):
    if not extract_folder.exists():
        print(f"Extracting {zip_path.name}...")
        with ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_folder)
    else:
        print(f"{extract_folder.name} already exists.")

loc_url = 'https://data.london.gov.uk/download/24rz6/77d9b319-931e-4090-bf8e-f578938bd352/LSOA2011%20AvPTAI2015.csv'
loc_path =  DATA_DIR / "location.csv"

crime_url = 'https://data.london.gov.uk/download/exy3m/vm7/MPS%20LSOA%20Level%20Crime%20(Historical).csv'
crime_path = DATA_DIR / "crime.csv"

deprivation_url = "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
deprivation_path = DATA_DIR / "deprivation.csv" 

house_url = "https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46/current/hpssadataset46medianpricepaidforresidentialpropertiesbylsoa.zip"
house_zip = DATA_DIR / "house_prices.zip"
house_path = DATA_DIR / "house_prices"


download_if_needed(loc_url,loc_path)
download_if_needed(crime_url,crime_path)
download_if_needed(deprivation_url,deprivation_path)
download_if_needed(house_url, house_zip)
extract_if_needed(house_zip, house_path)

location.csv already exists.
crime.csv already exists.
deprivation.csv already exists.
house_prices.zip already exists.
house_prices already exists.


In [3]:
# Load the data
excel_file = next(house_path.glob("*.xls"))

ptal_df = pd.read_csv('data/location.csv')
crime_df = pd.read_csv('data/crime.csv')
imd_df = pd.read_csv('data/deprivation.csv')
prices_df = pd.read_excel(
    excel_file,
    sheet_name="1a", # We specify sheet_name='1a' which contains the median prices.
    header=5
)

## Task 3: Exploratory Data Analysis (EDA)
Our first step in EDA is to understand the shape, data types, and missing values present in each dataset before we attempt to merge them.

In [4]:
print("Prices shape:", prices_df.shape)
print("Crime shape:", crime_df.shape)
print("IMD shape:", imd_df.shape)
print("PTAL shape:", ptal_df.shape)

Prices shape: (34753, 114)
Crime shape: (120796, 53)
IMD shape: (32844, 57)
PTAL shape: (4835, 5)


In [5]:
# Check basic info
print("--- HOUSING PRICES ---")
print(prices_df.info())
prices_df.head()

--- HOUSING PRICES ---
<class 'pandas.DataFrame'>
RangeIndex: 34753 entries, 0 to 34752
Columns: 114 entries, Local authority code to Year ending Mar 2023
dtypes: object(110), str(4)
memory usage: 30.2+ MB
None


,Local authority code,Local authority name,LSOA code,LSOA name,Year ending Dec 1995,Year ending Mar 1996,Year ending Jun 1996,Year ending Sep 1996,Year ending Dec 1996,Year ending Mar 1997,...,Year ending Dec 2020,Year ending Mar 2021,Year ending Jun 2021,Year ending Sep 2021,Year ending Dec 2021,Year ending Mar 2022,Year ending Jun 2022,Year ending Sep 2022,Year ending Dec 2022,Year ending Mar 2023
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,34750,34500,30500,30000,29950,29000,...,88000,81500,80500,89000,101500,94500,113000,97500,102500,106500
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,25000,25000,25300,25625,25000,24800,...,29750,33000,47000,49999,50159,50159,46000,43500,42000,43500
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,27000,27000,27250,28950,28500,28950,...,50000,51500,53000,58573.5,60000,62999,61499.5,60000,65499.5,66000
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,44500,44500,30000,26675,26000,25500,...,85000,:,83500,83000,80000,76000,75000,75000,70000,60000
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,22000,27000,27000,20600,20000,19500,...,:,:,:,95000,92500,95000,95000,92500,93750,92500


In [6]:
print("\n\n--- CRIME DATA ---")
print(crime_df.info())
crime_df.head()



--- CRIME DATA ---
<class 'pandas.DataFrame'>
RangeIndex: 120796 entries, 0 to 120795
Data columns (total 53 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   LSOA Code  120796 non-null  str  
 1   LSOA Name  120796 non-null  str  
 2   Borough    120796 non-null  str  
 3   Group      120796 non-null  str  
 4   SubGroup   120796 non-null  str  
 5   202007     120796 non-null  int64
 6   202008     120796 non-null  int64
 7   202009     120796 non-null  int64
 8   202010     120796 non-null  int64
 9   202011     120796 non-null  int64
 10  202012     120796 non-null  int64
 11  202101     120796 non-null  int64
 12  202102     120796 non-null  int64
 13  202103     120796 non-null  int64
 14  202104     120796 non-null  int64
 15  202105     120796 non-null  int64
 16  202106     120796 non-null  int64
 17  202107     120796 non-null  int64
 18  202108     120796 non-null  int64
 19  202109     120796 non-null  int64
 20  202110     12079

,LSOA Code,LSOA Name,Borough,Group,SubGroup,202007,202008,202009,202010,202011,...,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406
0,E01000005,City of London 001E,E09000001,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,E01000005,City of London 001E,E09000001,PUBLIC ORDER OFFENCES,PUBLIC FEAR ALARM OR DISTRESS,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,E01000005,City of London 001E,E09000001,THEFT,BICYCLE THEFT,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,E01000005,City of London 001E,E09000001,THEFT,OTHER THEFT,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,E01000005,City of London 001E,E09000001,THEFT,THEFT FROM THE PERSON,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [7]:
print("\n\n--- IMD DATA ---")
print(imd_df.info())
imd_df.head()



--- IMD DATA ---
<class 'pandas.DataFrame'>
RangeIndex: 32844 entries, 0 to 32843
Data columns (total 57 columns):
 #   Column                                                                                              Non-Null Count  Dtype  
---  ------                                                                                              --------------  -----  
 0   LSOA code (2011)                                                                                    32844 non-null  str    
 1   LSOA name (2011)                                                                                    32844 non-null  str    
 2   Local Authority District code (2019)                                                                32844 non-null  str    
 3   Local Authority District name (2019)                                                                32844 non-null  str    
 4   Index of Multiple Deprivation (IMD) Score                                                           3284

,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,...,16364,5,1.503,1615,1,1296,175,656,465,715
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,...,22676,7,1.196,2969,1,1156,182,580,394,620
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,...,17318,6,2.207,162,1,1350,146,759,445,804
3,E01000005,City of London 001E,E09000001,City of London,28.652,8678,3,0.211,6029,2,...,25218,8,1.769,849,1,1121,229,692,200,683
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,19.837,14486,5,0.117,14023,5,...,14745,5,0.969,4368,2,2040,522,1297,221,1285


In [8]:
print("\n\n--- PTAL DATA ---")
print(ptal_df.info())
ptal_df.head()



--- PTAL DATA ---
<class 'pandas.DataFrame'>
RangeIndex: 4835 entries, 0 to 4834
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   LSOA2011    4835 non-null   str    
 1   AvPTAI2015  4835 non-null   float64
 2   PTAL        4835 non-null   str    
 3   PTAIHigh    4835 non-null   float64
 4   PTAILow     4835 non-null   float64
dtypes: float64(3), str(2)
memory usage: 189.0 KB
None


,LSOA2011,AvPTAI2015,PTAL,PTAIHigh,PTAILow
0,E01000001,69.8233,6b,97.4435,35.9190
1,E01000002,83.7820,6b,117.9120,66.3503
2,E01000003,41.7417,6b,49.5318,37.3635
3,E01000005,85.8893,6b,120.8470,45.9168
4,E01000006,22.4558,5,34.1054,0.0000


### Converting Crime LSOA 2021 Codes to LSOA 2011 Codes

The crime dataset uses **LSOA 2021** boundary codes, while all other datasets use **LSOA 2011** codes. To enable a clean merge, we use the official **ONS Best Fit Lookup** (`LSOA11_LSOA21_LAD22_EW_LU_v2`) to convert the crime LSOA codes from 2021 to 2011 boundaries.

**Strategy:** Since each LSOA11 code maps to exactly one LSOA21 code, the conversion is straightforward.

In [9]:

# ---------------------------------------------------------------------------
# Load ONS LSOA 2011 -> LSOA 2021 Best Fit Lookup via ArcGIS FeatureServer
# This avoids file-generation API issues and always returns fresh data
# ---------------------------------------------------------------------------
LOOKUP_FS = (
    'https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services'
    '/LSOA11_LSOA21_LAD22_EW_LU_v2/FeatureServer/0/query'
    '?where=ObjectId>{offset}&orderByFields=ObjectId'
    '&outFields=ObjectId,LSOA11CD,LSOA21CD&f=json&resultRecordCount=2000'
)

lookup_path = DATA_DIR / 'lookup.csv'

if not lookup_path.exists():
    print('Downloading LSOA 2011 -> 2021 lookup via ArcGIS FeatureServer...')
    records, last_id = [], 0
    while True:
        url = LOOKUP_FS.format(offset=last_id)
        resp = urllib.request.urlopen(url)
        data = json.loads(resp.read())
        features = data.get('features', [])
        if not features:
            break
        for f in features:
            records.append([f['attributes']['LSOA11CD'], f['attributes']['LSOA21CD']])
            last_id = f['attributes']['ObjectId']
    with open(lookup_path, 'w', newline='', encoding='utf-8') as fh:
        writer = csv.writer(fh)
        writer.writerow(['LSOA11CD', 'LSOA21CD'])
        writer.writerows(records)
    print(f'Saved {len(records)} records to {lookup_path}')
else:
    print(f'lookup.csv already exists ({lookup_path})')

lu_df = pd.read_csv(lookup_path, usecols=['LSOA11CD', 'LSOA21CD'])
print(
    f'Lookup rows: {len(lu_df)}, '
    f'Unique LSOA21: {lu_df["LSOA21CD"].nunique()}, '
    f'Unique LSOA11: {lu_df["LSOA11CD"].nunique()}'
)


lookup.csv already exists (data\lookup.csv)
Lookup rows: 34753, Unique LSOA21: 34633, Unique LSOA11: 34753


The published ONS Best Fit lookup did not provide mappings for 181 London LSOA 2021 codes. To minimise data loss, these areas were allocated to 2011 LSOAs using the ONS Postcode Directory by assigning each LSOA 2021 to the 2011 LSOA containing the majority of its constituent postcodes.

In [10]:
# Iteratively map missing LSOA2021 -> LSOA2011 using the ONS Postcode Directory
ONSPD_FS = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services"
    "/ONS_Postcode_Directory_(February_2026)_for_the_UK_(Hosted_Table)"
    "/FeatureServer/0/query"
)

iteration = 1

while True:

    # Find remaining missing LSOA2021 codes
    crime_lsoa21 = set(crime_df["LSOA Code"].unique())
    lookup_lsoa21 = set(lu_df["LSOA21CD"].unique())
    missing_lsoa21 = sorted(crime_lsoa21 - lookup_lsoa21)

    print(f"\nIteration {iteration}")
    print(f"Missing LSOA2021 codes: {len(missing_lsoa21)}")

    if not missing_lsoa21:
        print("All LSOA2021 codes have been mapped.")
        break

    pcd_records = []
    batch_size = 25

    for i in range(0, len(missing_lsoa21), batch_size):

        batch = missing_lsoa21[i:i + batch_size]
        codes_str = "','".join(batch)

        where = f"lsoa21cd IN ('{codes_str}')"

        params = urllib.parse.urlencode({
            "where": where,
            "outFields": "lsoa21cd,lsoa11cd",
            "f": "json",
            "resultRecordCount": 10000
        })

        url = f"{ONSPD_FS}?{params}"

        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                data = json.loads(resp.read())

            for feat in data.get("features", []):
                attrs = feat["attributes"]
                pcd_records.append(
                    (attrs["lsoa21cd"], attrs["lsoa11cd"])
                )

        except Exception as e:
            print(f"Failed to query batch: {e}")

    print(f"Postcode records fetched: {len(pcd_records)}")

    # Count postcode evidence
    lsoa21_to_lsoa11_counts = defaultdict(Counter)

    for lsoa21, lsoa11 in pcd_records:
        if lsoa11:
            lsoa21_to_lsoa11_counts[lsoa21][lsoa11] += 1

    extra_rows = []

    for lsoa21, counts in lsoa21_to_lsoa11_counts.items():
        if counts:
            extra_rows.append({
                "LSOA21CD": lsoa21,
                "LSOA11CD": counts.most_common(1)[0][0]
            })

    mapped_missing = pd.DataFrame(extra_rows)

    print(f"New mappings found: {len(mapped_missing)}")

    if mapped_missing.empty:
        print("\nNo further mappings could be found.")
        print(f"Remaining unmapped codes ({len(missing_lsoa21)}):")
        print(missing_lsoa21)
        break

    # Avoid duplicates before appending
    mapped_missing = mapped_missing[
        ~mapped_missing["LSOA21CD"].isin(lu_df["LSOA21CD"])
    ]

    lu_df = pd.concat(
        [lu_df, mapped_missing],
        ignore_index=True
    ).drop_duplicates()

    iteration += 1

print("\nFinal lookup size:", len(lu_df))


Iteration 1
Missing LSOA2021 codes: 181
Postcode records fetched: 7034
New mappings found: 137

Iteration 2
Missing LSOA2021 codes: 44
Postcode records fetched: 1953
New mappings found: 38

Iteration 3
Missing LSOA2021 codes: 6
Postcode records fetched: 1000
New mappings found: 5

Iteration 4
Missing LSOA2021 codes: 1
Postcode records fetched: 18
New mappings found: 1

Iteration 5
Missing LSOA2021 codes: 0
All LSOA2021 codes have been mapped.

Final lookup size: 34934


In [11]:
# Monthly crime columns
crime_month_cols = [
    c for c in crime_df.columns
    if c not in ['LSOA Code', 'LSOA Name', 'Borough', 'Group', 'SubGroup']
]

# Merge crime_df with the lookup on LSOA21 code
print('\nMerging crime data with lookup table...')
crime_converted = crime_df.merge(
    lu_df,
    left_on='LSOA Code',
    right_on='LSOA21CD',
    how='left'
)

# Check how many crime rows could not be matched
unmatched = crime_converted['LSOA11CD'].isna().sum()
print(f'Crime rows with no LSOA11 match: {unmatched} out of {len(crime_converted)}')



Merging crime data with lookup table...


Crime rows with no LSOA11 match: 0 out of 121380


In [12]:
# Convert to LSOA 2011
crime_converted = (
    crime_converted
    .drop(columns=['LSOA Code', 'LSOA Name', 'LSOA21CD'])
    .rename(columns={'LSOA11CD': 'LSOA Code'})
)

cols_order = ['LSOA Code', 'Borough', 'Group', 'SubGroup'] + crime_month_cols
crime_converted = crime_converted[cols_order]

print(f'\nConverted crime_df shape: {crime_converted.shape}')
print(f'Unique LSOA 2011 codes: {crime_converted["LSOA Code"].nunique()}')

# Drop any remaining unmatched rows (if any)
n_before = len(crime_converted)
crime_converted = crime_converted.dropna(subset=['LSOA Code'])
n_after = len(crime_converted)
print(f'Dropped {n_before - n_after} rows with no valid LSOA mapping.')

print(crime_converted.head())



Converted crime_df shape: (121380, 52)
Unique LSOA 2011 codes: 4832
Dropped 0 rows with no valid LSOA mapping.
   LSOA Code    Borough                  Group  \
0  E01000005  E09000001               BURGLARY   
1  E01000005  E09000001  PUBLIC ORDER OFFENCES   
2  E01000005  E09000001                  THEFT   
3  E01000005  E09000001                  THEFT   
4  E01000005  E09000001                  THEFT   

                          SubGroup  202007  202008  202009  202010  202011  \
0  BURGLARY BUSINESS AND COMMUNITY       0       0       0       0       0   
1    PUBLIC FEAR ALARM OR DISTRESS       0       0       0       0       0   
2                    BICYCLE THEFT       0       0       0       0       0   
3                      OTHER THEFT       0       0       0       0       0   
4            THEFT FROM THE PERSON       0       0       0       0       0   

   202012  ...  202309  202310  202311  202312  202401  202402  202403  \
0       0  ...       0       0       0      

### Checking LSOA Code Discrepancies
Next, we will check the distinct LSOA codes in each dataset to see if they match and identify any discrepancies before merging.

In [13]:
# Extract unique LSOA codes from each dataframe (crime now uses converted LSOA11 codes)
prices_lsoa = set(prices_df['LSOA code'].dropna().unique())
crime_lsoa = set(crime_converted['LSOA Code'].dropna().unique())
imd_lsoa = set(imd_df['LSOA code (2011)'].dropna().unique())
ptal_lsoa = set(ptal_df['LSOA2011'].dropna().unique())

print(f"Unique LSOA codes in Prices: {len(prices_lsoa)}")
print(f"Unique LSOA codes in Crime (after LSOA11 conversion): {len(crime_lsoa)}")
print(f"Unique LSOA codes in IMD: {len(imd_lsoa)}")
print(f"Unique LSOA codes in PTAL: {len(ptal_lsoa)}")

# Find intersection of all LSOA codes
common_lsoa = prices_lsoa.intersection(crime_lsoa).intersection(imd_lsoa).intersection(ptal_lsoa)
print(f"\nLSOA codes present in ALL four datasets: {len(common_lsoa)}")

# Check discrepancies
print(f"\nDiscrepancies:")
print(f"Prices codes NOT in the common set: {len(prices_lsoa - common_lsoa)} (expected - covers all of England & Wales)")
print(f"Crime codes NOT in the common set: {len(crime_lsoa - common_lsoa)}")
print(f"IMD codes NOT in the common set: {len(imd_lsoa - common_lsoa)} (expected - covers all of England)")
print(f"PTAL codes NOT in the common set: {len(ptal_lsoa - common_lsoa)}")


Unique LSOA codes in Prices: 34753
Unique LSOA codes in Crime (after LSOA11 conversion): 4832
Unique LSOA codes in IMD: 32844
Unique LSOA codes in PTAL: 4835

LSOA codes present in ALL four datasets: 4832

Discrepancies:
Prices codes NOT in the common set: 29921 (expected - covers all of England & Wales)
Crime codes NOT in the common set: 0
IMD codes NOT in the common set: 28012 (expected - covers all of England)
PTAL codes NOT in the common set: 3


### Investigating the 3 Missing LSOAs

The discrepancy check found:
- **3 PTAL codes not in the common set** — these are also the exact same 3 LSOAs missing from the crime dataset.
- **All 3 belong to the City of London** (`E01000001`, `E01000002`, `E01000003` — City of London 001A/B/C), all rated PTAL 6b.
- These codes **do exist in both the Prices and IMD datasets**, so the issue is specific to the MPS crime data.

**Root cause:** The Metropolitan Police Service (MPS) does **not** cover the City of London — the City has its own separate police force (City of London Police). As a result, these 3 LSOAs are entirely absent from the MPS crime dataset by design, not due to a data error.

**Implication for the analysis:** When we perform our final inner merge on all four datasets, these 3 LSOAs will be dropped. This is acceptable given they represent a very small fraction of London (3 out of 4,835 LSOAs, ~0.06%) and the City of London is a unique, non-residential area that would likely be a statistical outlier in any model.

In [14]:
# Investigate the 3 PTAL codes not in the common set
ptal_not_common = ptal_lsoa - common_lsoa
print("The 3 PTAL/LSOA codes absent from the crime dataset:")
print(ptal_not_common)

# Show their details from the PTAL dataset
print("\nPTAL details:")
print(ptal_df[ptal_df['LSOA2011'].isin(ptal_not_common)][['LSOA2011', 'PTAL', 'AvPTAI2015']])

# Confirm presence in other datasets
print("\nPresence in each dataset:")
for code in sorted(ptal_not_common):
    print(f"  {code}: in Prices={code in prices_lsoa}, in IMD={code in imd_lsoa}, in Crime={code in crime_lsoa}")

# Load the lookup to show borough info
lu_full = pd.read_csv(DATA_DIR / "lookup.csv")
print("\nBorough details (from ONS lookup):")
print(lu_full[lu_full['LSOA11CD'].isin(ptal_not_common)][['LSOA11CD', 'LSOA11NM', 'LAD22NM']])


The 3 PTAL/LSOA codes absent from the crime dataset:
{'E01000003', 'E01000002', 'E01000001'}

PTAL details:
    LSOA2011 PTAL  AvPTAI2015
0  E01000001   6b     69.8233
1  E01000002   6b     83.7820
2  E01000003   6b     41.7417

Presence in each dataset:
  E01000001: in Prices=True, in IMD=True, in Crime=False
  E01000002: in Prices=True, in IMD=True, in Crime=False
  E01000003: in Prices=True, in IMD=True, in Crime=False



Borough details (from ONS lookup):
      LSOA11CD             LSOA11NM         LAD22NM
300  E01000001  City of London 001A  City of London
301  E01000002  City of London 001B  City of London
302  E01000003  City of London 001C  City of London
